# Modul 9: Feature Engineering Temporal & Machine Learning (XGBoost)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 9 (Minggu 10):** Merumuskan peramalan beban dari dataset `beban_cuaca_regional_168h.csv` sebagai *Supervised Learning* bebas *data leakage* (fitur lag, *rolling statistics*, *cyclical encoding* $\sin/\cos$) dan memvalidasinya dengan *TimeSeriesSplit*.

In [ ]:
# !pip install -q xgboost scikit-learn pandas numpy matplotlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost belum terpasang; menggunakan HistGradientBoostingRegressor (scikit-learn) sebagai fallback.")

plt.rcParams['axes.grid'] = True

## 1. Memuat Dataset Beban Regional & Rekayasa Fitur Kausal (Anti-Leakage)

In [ ]:
CSV_NAME = 'beban_cuaca_regional_168h.csv'
RAW_URL = 'https://raw.githubusercontent.com/novatha/deret-waktu-tee412/main/labs/datasets/' + CSV_NAME

def load_load_dataset():
    for p in [Path('datasets') / CSV_NAME, Path('labs/datasets') / CSV_NAME]:
        if p.exists():
            print(f"Memuat dataset lokal: {p}")
            return pd.read_csv(p, parse_dates=['Timestamp'], index_col='Timestamp')
    try:
        print(f"Mengunduh dataset dari GitHub: {RAW_URL}")
        return pd.read_csv(RAW_URL, parse_dates=['Timestamp'], index_col='Timestamp')
    except Exception as e:
        print(f"Fallback sintetis (alasan: {e})")
        rng = np.random.default_rng(10)
        n_hours = 24 * 7
        idx = pd.date_range('2026-09-07', periods=n_hours, freq='h')
        hr, dow = idx.hour, idx.dayofweek
        temp = 27 + 5 * np.sin(2 * np.pi * (hr - 8) / 24) + rng.normal(0, 1.2, n_hours)
        hum = 78 - 10 * np.sin(2 * np.pi * (hr - 8) / 24) + rng.normal(0, 2.0, n_hours)
        load = 300 + 70 * np.sin(2 * np.pi * (hr - 6) / 24) + 55 * np.exp(-((hr - 19) ** 2) / 8) - 25 * (dow >= 5) + 4.0 * temp + rng.normal(0, 6.0, n_hours)
        return pd.DataFrame({'Load_MW': load, 'Temp_C': temp, 'Humidity_Pct': hum, 'Is_Weekend': (dow >= 5).astype(int)}, index=idx)

df = load_load_dataset()

def create_causal_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    # Cyclical encoding jam & indikator akhir pekan
    d['hour_sin'] = np.sin(2 * np.pi * d.index.hour / 24.0)
    d['hour_cos'] = np.cos(2 * np.pi * d.index.hour / 24.0)
    d['is_weekend'] = d['Is_Weekend'] if 'Is_Weekend' in d.columns else (d.index.dayofweek >= 5).astype(int)
    # Lag & Rolling kausal (WAJIB .shift(1) sebelum .rolling()!)
    d['lag_1'] = d['Load_MW'].shift(1)
    d['lag_2'] = d['Load_MW'].shift(2)
    d['lag_24'] = d['Load_MW'].shift(24)
    d['roll_mean_24'] = d['Load_MW'].shift(1).rolling(24).mean()
    d['roll_std_24'] = d['Load_MW'].shift(1).rolling(24).std()
    return d.dropna()

df_feat = create_causal_features(df)
feature_cols = ['Temp_C', 'Humidity_Pct', 'hour_sin', 'hour_cos', 'is_weekend', 'lag_1', 'lag_2', 'lag_24', 'roll_mean_24', 'roll_std_24']
X, y = df_feat[feature_cols], df_feat['Load_MW']
print(f"Matriks fitur kausal siap: X={X.shape}, y={y.shape}")

## 2. Validasi Expanding Window (TimeSeriesSplit) & Evaluasi Out-of-Sample

In [ ]:
tscv = TimeSeriesSplit(n_splits=4, test_size=24)
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
    if HAS_XGB:
        model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
    else:
        model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    rmse = np.sqrt(mean_squared_error(y_va, pred))
    mae = mean_absolute_error(y_va, pred)
    fold_rmses.append(rmse)
    print(f"Fold {fold} (Val 24 jam) -> RMSE: {rmse:.2f} MW | MAE: {mae:.2f} MW")

print(f"Rata-rata RMSE Expanding Window CV: {np.mean(fold_rmses):.2f} MW")

plt.figure(figsize=(12, 4.5))
plt.plot(y_va.index, y_va.values, 'k-o', ms=4, lw=1.8, label='Aktual (Fold Terakhir 24 Jam)')
plt.plot(y_va.index, pred, color='#10b981', lw=2.0, label='Prediksi ML')
plt.title('Short-Term Load Forecasting (Dataset Regional) dengan Rekayasa Fitur Temporal')
plt.ylabel('Beban (MW)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Hapus `.shift(1)` pada `roll_mean_24` lalu jalankan ulang. Mengapa RMSE turun secara artifisial (*data leakage*)?
2. **[C5]** Plot *Feature Importance* model XGBoost dan analisis mengapa `lag_24`, `lag_1`, dan `Temp_C` sangat berpengaruh pada profil beban listrik!